In [46]:
import pandas as pd

df = pd.read_csv("subway_monthly_aggregates_2022/subway_30min_monthly_2022_01.csv")
df

,YEAR,MONTH,HOUR,HALF_HOUR,STATION_ID,LINE_NM,"STATION,_NM",GETON_CNT,GETOFF_CNT,gu_name,gu_code
0,2022,1,0,0,150,1호선,서울역,22,6,중구,11140.0
1,2022,1,0,0,151,1호선,시청,7,3,중구,11140.0
2,2022,1,0,0,152,1호선,종각,4,2,종로구,11110.0
3,2022,1,0,0,153,1호선,종로3가,31,7,종로구,11110.0
4,2022,1,0,0,154,1호선,종로5가,4,8,종로구,11110.0
...,...,...,...,...,...,...,...,...,...,...,...
12920,2022,2,0,0,2716,7호선,중계,0,1,노원구,11350.0
12921,2022,2,0,0,2722,7호선,상봉(시외버스터미널),1,0,중랑구,11260.0
12922,2022,2,0,0,2723,7호선,면목,1,1,중랑구,11260.0
12923,2022,2,0,0,2818,8호선,가락시장,0,1,송파구,11710.0


In [43]:
import pandas as pd
import os

# 1. 대상 폴더 경로 설정
folder_path = 'subway_monthly_aggregates_2024'

# 2. 폴더 내의 모든 파일 순회
for filename in os.listdir(folder_path):
    # 확장자가 csv인 파일만 선택
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        
        # CSV 읽기
        df = pd.read_csv(file_path)
        
        # 3. 모든 컬럼을 순회하며 백틱(`) 제거
        # object(문자열) 타입인 컬럼에 대해서만 실행
        for col in df.columns:
            if df[col].dtype == 'str':
                df[col] = df[col].str.replace("`", "", regex=False)
        
        # 4. 결과 저장 (기존 파일 덮어쓰기 또는 새 이름으로 저장)
        df.to_csv(file_path, index=False, encoding='utf-8-sig')
        print(f"처리 완료: {filename}")

처리 완료: subway_30min_monthly_2024_01.csv
처리 완료: subway_30min_monthly_2024_02.csv
처리 완료: subway_30min_monthly_2024_03.csv
처리 완료: subway_30min_monthly_2024_04.csv
처리 완료: subway_30min_monthly_2024_05.csv
처리 완료: subway_30min_monthly_2024_06.csv
처리 완료: subway_30min_monthly_2024_07.csv
처리 완료: subway_30min_monthly_2024_08.csv
처리 완료: subway_30min_monthly_2024_09.csv
처리 완료: subway_30min_monthly_2024_10.csv
처리 완료: subway_30min_monthly_2024_11.csv
처리 완료: subway_30min_monthly_2024_12.csv


## 자치구별 월별 승하차 인원 집계 (2022~2024)

In [47]:
import pandas as pd
import glob

# 2022~2024 전체 파일 로드
folders = [
    'subway_monthly_aggregates_2022',
    'subway_monthly_aggregates_2023',
    'subway_monthly_aggregates_2024',
]

df_all = pd.concat(
    [pd.read_csv(f, encoding='utf-8-sig') for folder in folders for f in sorted(glob.glob(f'{folder}/*.csv'))],
    ignore_index=True
)

print(f"전체 행: {len(df_all):,}  |  기간: {df_all['YEAR'].min()}-{df_all['MONTH'].min():02d} ~ {df_all['YEAR'].max()}-{df_all['MONTH'].max():02d}")
df_all.head(3)

전체 행: 512,974  |  기간: 2022-01 ~ 2025-12


,YEAR,MONTH,HOUR,HALF_HOUR,STATION_ID,LINE_NM,"STATION,_NM",GETON_CNT,GETOFF_CNT,gu_name,gu_code
0,2022,1,0,0,150,1호선,서울역,22,6,중구,11140.0
1,2022,1,0,0,151,1호선,시청,7,3,중구,11140.0
2,2022,1,0,0,152,1호선,종각,4,2,종로구,11110.0


In [48]:
# 서울 내 역만 필터 (gu_name이 있는 행)
df_seoul = df_all.dropna(subset=['gu_name']).copy()

# 자치구 × 연도 × 월 그룹바이 → 총 승하차 집계
gu_monthly = (
    df_seoul
    .groupby(['YEAR', 'MONTH', 'gu_code', 'gu_name'], as_index=False)
    .agg(
        total_geton  = ('GETON_CNT',  'sum'),
        total_getoff = ('GETOFF_CNT', 'sum'),
    )
)
gu_monthly['total_passenger'] = gu_monthly['total_geton'] + gu_monthly['total_getoff']
gu_monthly = gu_monthly.sort_values(['YEAR', 'MONTH', 'gu_name']).reset_index(drop=True)

print(f"집계 행: {len(gu_monthly):,}  (25개 자치구 × 36개월 = 최대 900행)")
gu_monthly

집계 행: 925  (25개 자치구 × 36개월 = 최대 900행)


,YEAR,MONTH,gu_code,gu_name,total_geton,total_getoff,total_passenger
0,2022,1,11680.0,강남구,9708350,9683576,19391926
1,2022,1,11740.0,강동구,3480387,3317290,6797677
2,2022,1,11305.0,강북구,1629651,1873908,3503559
3,2022,1,11500.0,강서구,2679278,3464455,6143733
4,2022,1,11620.0,관악구,3406869,3224837,6631706
...,...,...,...,...,...,...,...
920,2025,1,11170.0,용산구,1268,1126,2394
921,2025,1,11380.0,은평구,522,1531,2053
922,2025,1,11110.0,종로구,8371,1829,10200
923,2025,1,11140.0,중구,12988,4735,17723


In [49]:
# 결과 저장
gu_monthly.to_csv('subway_gu_monthly_2022_2024.csv', index=False, encoding='utf-8-sig')
print("저장 완료: subway_gu_monthly_2022_2024.csv")

# 자치구별 3년 합계 요약
gu_summary = (
    gu_monthly
    .groupby('gu_name', as_index=False)
    .agg(total_geton=('total_geton','sum'), total_getoff=('total_getoff','sum'), total_passenger=('total_passenger','sum'))
    .sort_values('total_passenger', ascending=False)
    .reset_index(drop=True)
)
print("\n[자치구별 2022~2024 승하차 합계]")
gu_summary

저장 완료: subway_gu_monthly_2022_2024.csv

[자치구별 2022~2024 승하차 합계]


,gu_name,total_geton,total_getoff,total_passenger
0,강남구,386669565,383200350,769869915
1,중구,289698314,282059098,571757412
2,송파구,267399969,248961191,516361160
3,마포구,238380181,231160828,469541009
4,서초구,200504858,201616705,402121563
5,종로구,198108931,188003097,386112028
6,광진구,183598044,175988299,359586343
7,동작구,159501162,170122880,329624042
8,노원구,152363897,145433997,297797894
9,성동구,145077201,143205567,288282768


In [51]:
# ── 자치구별 월별 승하차 Wide 포맷 (202201_geton / 202201_getoff ~ 202412) ──

# YYYYMM 키 생성
gu_monthly['ym'] = gu_monthly['YEAR'].astype(str) + gu_monthly['MONTH'].astype(str).str.zfill(2)

# 승차 / 하차 각각 피벗
pivot_geton  = gu_monthly.pivot_table(index=['gu_code','gu_name'], columns='ym', values='total_geton',  aggfunc='sum')
pivot_getoff = gu_monthly.pivot_table(index=['gu_code','gu_name'], columns='ym', values='total_getoff', aggfunc='sum')

pivot_geton.columns  = [f'{ym}_geton'  for ym in pivot_geton.columns]
pivot_getoff.columns = [f'{ym}_getoff' for ym in pivot_getoff.columns]

# 월 순서대로 geton/getoff 교차 정렬
months = sorted(gu_monthly['ym'].unique())
cols_ordered = [col for ym in months for col in (f'{ym}_geton', f'{ym}_getoff')]

wide = (
    pd.concat([pivot_geton, pivot_getoff], axis=1)[cols_ordered]
    .reset_index()
)
wide['gu_code'] = wide['gu_code'].astype(int)
wide = wide.sort_values('gu_code').reset_index(drop=True)

# 저장
out_path = 'subway_gu_wide_202201_202412.csv'
wide.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"저장 완료: {out_path}")
print(f"Shape: {wide.shape}  →  자치구 {len(wide)}개 × 컬럼 {wide.shape[1]}개 (gu_code + gu_name + 월×2)")
wide

저장 완료: subway_gu_wide_202201_202412.csv
Shape: (25, 76)  →  자치구 25개 × 컬럼 76개 (gu_code + gu_name + 월×2)


,gu_code,gu_name,202201_geton,202201_getoff,202202_geton,202202_getoff,202203_geton,202203_getoff,202204_geton,202204_getoff,...,202409_geton,202409_getoff,202410_geton,202410_getoff,202411_geton,202411_getoff,202412_geton,202412_getoff,202501_geton,202501_getoff
0,11110,종로구,4342692,4083059,3646743,3436870,4166090,3940136,4768867,4526935,...,5484708,5238690,6427553,6140873,6342368,6024876,6359508,6032470,8371,1829
1,11140,중구,6034978,5843985,5042615,4870897,5760383,5601476,6555818,6420654,...,8353571,8150434,9582007,9343492,9617975,9368612,9655623,9386192,12988,4735
2,11170,용산구,1675761,1638414,1400655,1370633,1613398,1581310,1837871,1828038,...,2182922,2183468,2511724,2519882,2460847,2459428,2507213,2492699,1268,1126
3,11200,성동구,3350261,3305764,2851007,2822592,3360305,3324499,3738344,3678613,...,4108215,4061634,4666333,4607600,4697508,4631648,4701712,4645796,1084,2321
4,11215,광진구,4397778,4249706,3750580,3608749,4329291,4161777,4938785,4756844,...,4938619,4716574,5535158,5270590,5478864,5232405,5431670,5196925,2230,2658
5,11230,동대문구,1259464,1184241,1037399,974432,1189790,1116220,1308521,1199877,...,1425512,1311212,1595658,1462714,1574855,1441563,1574504,1447150,164,1138
6,11260,중랑구,1804357,1729566,1553239,1491808,1768886,1691093,1924228,1819660,...,1905480,1823713,2110171,2020382,2108338,2021805,2125130,2037222,342,753
7,11290,성북구,2217254,2263780,1907124,1964473,2302329,2342899,2514161,2529064,...,2777243,2774439,3046058,3046637,3073656,3075383,2985453,2995716,770,1560
8,11305,강북구,1629651,1873908,1409846,1622942,1605762,1842893,1723427,1974654,...,1780057,2033879,1960897,2257969,1958791,2262075,1982424,2273046,426,810
9,11320,도봉구,1300628,1287171,1113918,1103632,1258592,1247753,1365814,1340702,...,1428790,1409695,1580482,1563520,1580527,1563675,1585763,1565238,170,601
